# NB03 — Evaluasi Hasil Clustering (FAISS + HDBSCAN)

**Tujuan:** Evaluasi kuantitatif kualitas clustering dari NB02 menggunakan tiga metrik:
- **Coverage & Noise Rate** — seberapa banyak wajah berhasil di-cluster
- **Silhouette Score** — separasi antar cluster (GPU-accelerated via FAISS)
- **DBCV** (Density-Based Clustering Validation) — metrik native untuk density-based clustering
- **Davies-Bouldin Index** — rasio dispersi dalam cluster vs antar cluster

**Input:** `output_nb01/embeddings.npy`, `output_nb02/labels.npy`, `output_nb02/metadata_labeled.pkl`  
**Output:** `output_nb03/evaluasi_summary.pkl`, `output_nb03/*.png`

---

> **GPU digunakan untuk:** komputasi pairwise cosine distance matrix (O(N²)) via FAISS IndexFlatIP  
> → dipakai bersama oleh Silhouette Score dan DBCV (subsample 3000 titik)


## 0. Instalasi & Import

In [ ]:
import subprocess, sys

def install_faiss():
    for pkg in ["faiss-gpu-cu12", "faiss-gpu-cu11", "faiss-cpu"]:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", pkg, "-q"],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f"✅ Berhasil install: {pkg}")
            return pkg
        if result.stderr:
            print(f"  ↳ {pkg} gagal: {result.stderr[:200]}")
    raise RuntimeError("Gagal install faiss — coba manual: pip install faiss-cpu")

try:
    import faiss
    print(f"✅ FAISS sudah ada: {faiss.__version__}")
except ImportError:
    install_faiss()

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "hdbscan", "umap-learn", "scikit-learn", "matplotlib", "pillow", "-q"],
    check=True
)


In [ ]:
import os
import pickle
import time
import warnings
from pathlib import Path
from collections import Counter

import faiss
import hdbscan
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import umap

from sklearn.metrics import silhouette_score, davies_bouldin_score
from hdbscan.validity import validity_index

warnings.filterwarnings('ignore')

print(f"faiss   version : {faiss.__version__}")
print(f"hdbscan version : {hdbscan.__version__}")

try:
    ngpu = faiss.get_num_gpus()
    print(f"FAISS GPU count : {ngpu}  {'(GPU aktif ✅)' if ngpu > 0 else '(hanya CPU ⚠️)'}")
    if ngpu == 0:
        print("  ⚠️  Aktifkan GPU runtime: Runtime → Change runtime type → T4 GPU")
except Exception:
    print("FAISS GPU: tidak tersedia")


## 1. Konfigurasi

In [ ]:
# Jika pakai Google Drive — jalankan cell ini
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
# ─── SESUAIKAN PATH INI ───────────────────────────────────────────────────────
INPUT_EMB_DIR  = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings/output_nb01")  # output NB01
INPUT_CLUS_DIR = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings/output_nb02")  # output NB02
OUTPUT_DIR     = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings/output_nb03")  # output NB03
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Konstanta evaluasi ───────────────────────────────────────────────────────
SUBSAMPLE    = 3000   # titik untuk Silhouette & DBCV (O(N²) → subsample wajib)
RANDOM_SEED  = 42

print(f"Input embeddings : {INPUT_EMB_DIR}")
print(f"Input clustering : {INPUT_CLUS_DIR}")
print(f"Output dir       : {OUTPUT_DIR}")
print(f"SUBSAMPLE        : {SUBSAMPLE}")


## 2. Load Data

In [ ]:
# ── Load embeddings (dari NB01) ────────────────────────────────────────────────
embeddings = np.load(INPUT_EMB_DIR / "embeddings.npy").astype(np.float32)
N, D = embeddings.shape

# ── Load labels & metadata (dari NB02) ────────────────────────────────────────
labels = np.load(INPUT_CLUS_DIR / "labels.npy")

with open(INPUT_CLUS_DIR / "metadata_labeled.pkl", "rb") as f:
    metadata = pickle.load(f)

with open(INPUT_CLUS_DIR / "cluster_summary.pkl", "rb") as f:
    cluster_summary = pickle.load(f)

print(f"Embeddings shape  : {embeddings.shape}  dtype={embeddings.dtype}")
print(f"Labels shape      : {labels.shape}  dtype={labels.dtype}")
print(f"Metadata records  : {len(metadata)}")
print()
print(f"Cluster summary dari NB02:")
print(f"  Jumlah cluster  : {cluster_summary['n_clusters']}")
print(f"  Coverage        : {cluster_summary['coverage_pct']:.1f}%")
print(f"  Noise           : {cluster_summary['n_noise']:,}  ({cluster_summary['n_noise']/N*100:.1f}%)")
print(f"  Params          : {cluster_summary['params']}")


## 3. Metrik Evaluasi Kuantitatif

### 3a. Coverage & Noise Rate


In [ ]:
n_clusters  = cluster_summary['n_clusters']
n_noise     = int((labels == -1).sum())
n_clustered = int((labels >= 0).sum())
coverage    = n_clustered / N * 100
noise_pct   = n_noise / N * 100

cluster_sizes = Counter(labels[labels >= 0])
sizes = sorted(cluster_sizes.values(), reverse=True)

print(f"Coverage rate     : {coverage:.2f}%  ({n_clustered:,} / {N:,} wajah)")
print(f"Noise rate        : {noise_pct:.2f}%  ({n_noise:,} wajah)")
print(f"Jumlah cluster    : {n_clusters}")
print(f"Ukuran cluster    : min={min(sizes)}, max={max(sizes)}, mean={np.mean(sizes):.1f}, median={np.median(sizes):.0f}")


### 3b. Silhouette Score (GPU-accelerated via FAISS)

Silhouette Score mengukur seberapa mirip tiap titik dengan cluster-nya sendiri
dibanding cluster tetangga. Range: **[-1, 1]**, makin tinggi makin baik.

Komputasi distance matrix menggunakan **FAISS GPU** (IndexFlatIP) pada subsample
untuk menghindari O(N²) bottleneck.


In [ ]:
t0 = time.time()

# Subsample dari titik yang ter-cluster (bukan noise)
rng = np.random.default_rng(seed=RANDOM_SEED)
idx_pool = np.where(labels >= 0)[0]
n_sub    = min(SUBSAMPLE, len(idx_pool))
idx_sub  = rng.choice(idx_pool, size=n_sub, replace=False)
idx_sub  = np.sort(idx_sub)

emb_sub = embeddings[idx_sub].copy().astype(np.float32)
faiss.normalize_L2(emb_sub)

# ── Bangun pairwise cosine distance matrix via FAISS GPU ──────────────────────
try:
    # GPU
    res   = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, faiss.IndexFlatIP(D))
    print(f"Menggunakan FAISS GPU untuk distance matrix ({n_sub}×{n_sub})...")
except Exception:
    # CPU fallback
    index = faiss.IndexFlatIP(D)
    print(f"Menggunakan FAISS CPU untuk distance matrix ({n_sub}×{n_sub})...")

index.add(emb_sub)
sim_matrix, _ = index.search(emb_sub, n_sub)   # pairwise cosine similarity

# cosine distance = 1 - cosine similarity, clip ke [0, 2]
dist_matrix = np.clip(1.0 - sim_matrix.astype(np.float64), 0.0, 2.0)
np.fill_diagonal(dist_matrix, 0.0)

t_dist = time.time() - t0
print(f"Distance matrix selesai ({t_dist:.2f} detik)  shape={dist_matrix.shape}")

# ── Silhouette Score ──────────────────────────────────────────────────────────
labels_sub = labels[idx_sub]
sil_score  = silhouette_score(dist_matrix, labels_sub, metric='precomputed')

t_sil = time.time() - t0
print(f"
Silhouette Score  : {sil_score:.4f}  (subsample {n_sub:,} titik, {t_sil:.1f} detik)")
print(f"  Range [-1, 1] — makin tinggi makin baik")
print(f"  > 0.5  = cluster terpisah jelas")
print(f"  > 0.25 = cluster cukup baik")
print(f"  < 0    = titik mungkin salah cluster")


### 3c. DBCV — Density-Based Clustering Validation

DBCV dirancang khusus untuk **density-based clustering** seperti HDBSCAN.
Berbeda dengan Silhouette, DBCV dapat menilai cluster berbentuk arbitrary (non-convex)
yang umum pada data wajah nyata.

Range: **[-1, 1]**, makin tinggi makin baik.  
Distance matrix dari 3b di-reuse (sudah dihitung di GPU).


In [ ]:
t0 = time.time()

# Filter noise dari subsample untuk DBCV
mask_valid = labels_sub >= 0
dist_sub   = dist_matrix[mask_valid][:, mask_valid].astype(np.float64)
labels_valid = labels_sub[mask_valid]

print(f"Menghitung DBCV ({mask_valid.sum()} titik non-noise, metric=precomputed)...")

dbcv_score = validity_index(
    dist_sub,
    labels_valid,
    metric='precomputed',
    d=D          # dimensi asli embedding (512) untuk normalisasi DBCV internal
)

t_dbcv = time.time() - t0
print(f"
DBCV Score        : {dbcv_score:.4f}  ({t_dbcv:.1f} detik)")
print(f"  Range [-1, 1] — makin tinggi makin baik")
print(f"  Metrik native density-based clustering")
print(f"  Lebih akurat dari Silhouette untuk cluster non-convex (wajah)")


### 3d. Davies-Bouldin Index

In [ ]:
t0 = time.time()

# Davies-Bouldin menggunakan embedding langsung (cosine distance)
# sklearn menghitung di CPU, subsample sama
emb_sub_f64 = embeddings[idx_sub][mask_valid].astype(np.float64)

dbi_score = davies_bouldin_score(emb_sub_f64, labels_valid)

t_dbi = time.time() - t0
print(f"Davies-Bouldin Index : {dbi_score:.4f}  ({t_dbi:.1f} detik)")
print(f"  Range [0, ∞) — makin RENDAH makin baik")
print(f"  < 1.0 = cluster baik")


### 3e. Ringkasan Semua Metrik

In [ ]:
print("=" * 52)
print("  RINGKASAN METRIK EVALUASI NB03")
print("=" * 52)
print(f"  Dataset           : {N:,} wajah × {D} dimensi")
print(f"  Subsample         : {n_sub:,} titik (seed={RANDOM_SEED})")
print()
print(f"  Coverage rate     : {coverage:.2f}%")
print(f"  Noise rate        : {noise_pct:.2f}%")
print(f"  Jumlah cluster    : {n_clusters}")
print()
print(f"  Silhouette Score  : {sil_score:.4f}   (range [-1,1], ↑ lebih baik)")
print(f"  DBCV Score        : {dbcv_score:.4f}   (range [-1,1], ↑ lebih baik)")
print(f"  Davies-Bouldin    : {dbi_score:.4f}   (range [0,∞),  ↓ lebih baik)")
print("=" * 52)


## 4. Visualisasi 2D (UMAP)

UMAP dipakai **hanya untuk visualisasi** — bukan clustering.  
Reduksi 512 → 2 dimensi dengan cosine metric.


In [ ]:
print("Menjalankan UMAP 2D (512 → 2 dim)...")
t0 = time.time()

reducer = umap.UMAP(
    n_components=2,
    metric='cosine',
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=False
)
emb_2d = reducer.fit_transform(embeddings)

t_umap = time.time() - t0
print(f"UMAP selesai ({t_umap:.1f} detik)  shape={emb_2d.shape}")

# ── Plot scatter ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 10))

# Noise points (abu-abu, transparan)
noise_mask = labels == -1
ax.scatter(
    emb_2d[noise_mask, 0], emb_2d[noise_mask, 1],
    c='#CBD5E1', s=2, alpha=0.3, label=f'Noise ({noise_mask.sum():,})', rasterized=True
)

# Cluster points (warna per cluster, top-30 cluster paling besar)
top_ids = [c[0] for c in cluster_sizes.most_common(30)]
cmap    = plt.cm.get_cmap('tab20', len(top_ids))

for i, cid in enumerate(top_ids):
    mask = labels == cid
    ax.scatter(
        emb_2d[mask, 0], emb_2d[mask, 1],
        c=[cmap(i)], s=4, alpha=0.7, rasterized=True
    )

# Cluster kecil (tidak masuk top-30) — warna gelap
other_mask = (labels >= 0) & (~np.isin(labels, top_ids))
if other_mask.sum() > 0:
    ax.scatter(
        emb_2d[other_mask, 0], emb_2d[other_mask, 1],
        c='#374151', s=2, alpha=0.4, label=f'Cluster lain ({other_mask.sum():,})', rasterized=True
    )

ax.set_title(
    f"UMAP 2D — {N:,} wajah  |  {n_clusters} cluster  |  "
    f"Coverage {coverage:.1f}%  |  Silhouette {sil_score:.3f}  |  DBCV {dbcv_score:.3f}",
    fontsize=11, fontweight='bold'
)
ax.set_xlabel("UMAP dim 1", fontsize=10)
ax.set_ylabel("UMAP dim 2", fontsize=10)
ax.legend(loc='upper right', fontsize=8, markerscale=3)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "umap_2d_clustering.png", dpi=150, bbox_inches='tight')
plt.show()
print("Tersimpan: umap_2d_clustering.png")


## 5. Distribusi Cluster

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ── 5a. Pie: clustered vs noise ───────────────────────────────────────────────
axes[0].pie(
    [n_clustered, n_noise],
    labels=[f"Clustered\n({n_clustered:,})", f"Noise\n({n_noise:,})"],
    colors=['#4F46E5', '#94A3B8'],
    autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10}
)
axes[0].set_title("Komposisi Hasil Clustering", fontsize=12, fontweight='bold')

# ── 5b. Bar chart top-20 cluster ─────────────────────────────────────────────
top20 = cluster_sizes.most_common(20)
axes[1].bar(range(20), [s for _, s in top20], color='#4F46E5', edgecolor='white')
axes[1].set_xticks(range(0, 20, 2))
axes[1].set_xticklabels([str(top20[i][0]) for i in range(0, 20, 2)], rotation=45, ha='right')
axes[1].set_xlabel("Cluster ID", fontsize=10)
axes[1].set_ylabel("Jumlah Wajah", fontsize=10)
axes[1].set_title("Top 20 Cluster Terbesar", fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

# ── 5c. Cumulative coverage ───────────────────────────────────────────────────
cumsum_pct = np.cumsum(sizes) / N * 100
axes[2].plot(range(1, len(sizes) + 1), cumsum_pct, color='#4F46E5', linewidth=2)
axes[2].axhline(80, color='#F59E0B', linewidth=1.5, linestyle='--', label='80%')
axes[2].axhline(90, color='#EF4444', linewidth=1.5, linestyle='--', label='90%')
top80 = int(np.searchsorted(cumsum_pct, 80)) + 1
top90 = int(np.searchsorted(cumsum_pct, 90)) + 1
axes[2].set_xlabel("Jumlah Cluster (urut terbesar)", fontsize=10)
axes[2].set_ylabel("Kumulatif % Embedding", fontsize=10)
axes[2].set_title("Cumulative Coverage", fontsize=12, fontweight='bold')
axes[2].legend(fontsize=9)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "distribusi_cluster.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Top {top80} cluster mencakup 80% embedding")
print(f"Top {top90} cluster mencakup 90% embedding")


## 6. Sample Face Crops — Top Cluster

In [ ]:
def crop_face(photo_path, bbox, padding=0.25, size=112):
    try:
        img = Image.open(photo_path).convert("RGB")
        W, H = img.size
        x1, y1, x2, y2 = bbox
        bw, bh = x2 - x1, y2 - y1
        x1 = max(0, x1 - bw * padding);  y1 = max(0, y1 - bh * padding)
        x2 = min(W, x2 + bw * padding);  y2 = min(H, y2 + bh * padding)
        return img.crop((x1, y1, x2, y2)).resize((size, size))
    except Exception:
        return Image.new('RGB', (size, size), color=(200, 200, 200))

N_SHOW_CLUSTERS = 10
N_SHOW_PER_ROW  = 8

top_ids = [c[0] for c in cluster_sizes.most_common(N_SHOW_CLUSTERS)]
meta_by_idx = {i: m for i, m in enumerate(metadata)}

fig, axes = plt.subplots(N_SHOW_CLUSTERS, N_SHOW_PER_ROW,
                         figsize=(N_SHOW_PER_ROW * 1.5, N_SHOW_CLUSTERS * 1.6))
fig.suptitle(f"Sample Wajah dari Top {N_SHOW_CLUSTERS} Cluster Terbesar",
             fontsize=13, fontweight='bold', y=1.01)

for row, cid in enumerate(top_ids):
    cluster_idx = np.where(labels == cid)[0]
    rng2 = np.random.default_rng(seed=42 + cid)
    sample_idx = rng2.choice(cluster_idx, size=min(N_SHOW_PER_ROW, len(cluster_idx)), replace=False)
    for col in range(N_SHOW_PER_ROW):
        ax = axes[row, col]
        if col < len(sample_idx):
            m = meta_by_idx[sample_idx[col]]
            ax.imshow(crop_face(m['photo_path'], m['bbox']))
            if col == 0:
                ax.set_ylabel(f"C{cid}\n({len(cluster_idx)})",
                              fontsize=8, rotation=0, labelpad=35, va='center')
        else:
            ax.set_facecolor('#F1F5F9')
        ax.axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sample_cluster_faces.png", dpi=150, bbox_inches='tight')
plt.show()
print("Tersimpan: sample_cluster_faces.png")


## 7. Analisis Noise

In [ ]:
noise_meta     = [metadata[i] for i in range(N) if labels[i] == -1]
clustered_meta = [metadata[i] for i in range(N) if labels[i] >= 0]

noise_det_scores     = [m['det_score'] for m in noise_meta]
clustered_det_scores = [m['det_score'] for m in clustered_meta]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── 7a. Distribusi det_score: noise vs clustered ──────────────────────────────
axes[0].hist(clustered_det_scores, bins=40, alpha=0.7, color='#4F46E5',
             label=f'Clustered (n={len(clustered_det_scores):,})', density=True)
axes[0].hist(noise_det_scores, bins=40, alpha=0.7, color='#EF4444',
             label=f'Noise (n={len(noise_det_scores):,})', density=True)
axes[0].set_xlabel("Detection Score", fontsize=11)
axes[0].set_ylabel("Densitas", fontsize=11)
axes[0].set_title("Distribusi Detection Score
Noise vs Clustered", fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# ── 7b. Top subfolder penyumbang noise terbanyak ─────────────────────────────
noise_folders = Counter(
    Path(m['photo_path']).parent.name for m in noise_meta
)
top_folders = noise_folders.most_common(15)
if top_folders:
    names  = [t[0][:20] for t in top_folders]
    counts = [t[1] for t in top_folders]
    axes[1].barh(range(len(names)), counts, color='#EF4444', edgecolor='white')
    axes[1].set_yticks(range(len(names)))
    axes[1].set_yticklabels(names, fontsize=9)
    axes[1].set_xlabel("Jumlah Noise Wajah", fontsize=11)
    axes[1].set_title("Top Subfolder Penyumbang Noise", fontsize=12, fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'Tidak ada subfolder info', ha='center', va='center')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "analisis_noise.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Noise det_score — mean: {np.mean(noise_det_scores):.3f}, "
      f"median: {np.median(noise_det_scores):.3f}")
print(f"Clustered det_score — mean: {np.mean(clustered_det_scores):.3f}, "
      f"median: {np.median(clustered_det_scores):.3f}")


## 8. Simpan Output

In [ ]:
evaluasi_summary = {
    "dataset": {
        "n_embeddings"  : N,
        "n_dimensions"  : D,
        "subsample"     : n_sub,
        "random_seed"   : RANDOM_SEED,
    },
    "metrics": {
        "coverage_pct"   : coverage,
        "noise_pct"      : noise_pct,
        "n_clusters"     : n_clusters,
        "silhouette"     : float(sil_score),
        "dbcv"           : float(dbcv_score),
        "davies_bouldin" : float(dbi_score),
    },
    "cluster_stats": {
        "min_size"   : int(min(sizes)),
        "max_size"   : int(max(sizes)),
        "mean_size"  : float(np.mean(sizes)),
        "median_size": float(np.median(sizes)),
    },
    "nb02_params": cluster_summary['params'],
}

summary_path = OUTPUT_DIR / "evaluasi_summary.pkl"
with open(summary_path, "wb") as f:
    pickle.dump(evaluasi_summary, f)

print(f"Tersimpan: {summary_path.name}  ({summary_path.stat().st_size / 1024:.1f} KB)")


## 9. Ringkasan Akhir

In [ ]:
print("=" * 56)
print("  RINGKASAN NB03 — EVALUASI CLUSTERING")
print("=" * 56)
print(f"  Dataset              : {N:,} wajah × {D} dimensi")
print(f"  Subsample evaluasi   : {n_sub:,} titik")
print()
print(f"  [Hasil Clustering]")
print(f"    Jumlah cluster     : {n_clusters}")
print(f"    Coverage           : {coverage:.2f}%")
print(f"    Noise              : {noise_pct:.2f}%")
print()
print(f"  [Metrik Kualitas]")
print(f"    Silhouette Score   : {sil_score:.4f}   ↑ (range [-1, 1])")
print(f"    DBCV Score         : {dbcv_score:.4f}   ↑ (range [-1, 1])")
print(f"    Davies-Bouldin     : {dbi_score:.4f}   ↓ (range [0, ∞))")
print()
print(f"  [Output]")
print(f"    evaluasi_summary.pkl")
print(f"    umap_2d_clustering.png")
print(f"    distribusi_cluster.png")
print(f"    sample_cluster_faces.png")
print(f"    analisis_noise.png")
print("=" * 56)
print()
print("➜  Lanjut ke NB04 (jika ada hyperparameter tuning)")
